# Dia 15 — Coleta de informações de domínio

**🃏 Cartas:** 15 · 36 · 57 &nbsp;·&nbsp; **📘 Original:** *Day 15. Domain information gathering*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 15 — Quem registrou o domínio?

Sexta-feira, 7h40. A sala do NFA ainda cheira a café requentado quando a Iara joga um post-it amarelo
na sua mesa: **`megaoferta-brasil`**.

— Todo golpe online precisa de um endereço. Pode trocar de nome, de chave PIX, de número de
WhatsApp. Mas alguém, em algum momento, **registrou esse domínio**. E registro deixa data.

Ela puxa uma cadeira.

— Tem gente que ainda chama isso só de WHOIS. Funciona, mas é texto solto, cada servidor responde de
um jeito. Hoje existe o **RDAP**, que devolve tudo em JSON, organizado. Eu quero os dois: um confirma
o outro. Se discordarem, isso também é informação.

Você abre o terminal. A pergunta do dia é simples e poderosa: **quantos dias de vida o site tinha
quando fez a primeira vítima?** Se a resposta for "menos de duas semanas", a Delegada Rita vai
querer saber disso antes do almoço.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-15-dominios")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Rodar os exemplos originais com `dnspython` e `python-whois`.
- Entender a diferença entre **WHOIS** (texto, porta 43) e **RDAP** (JSON, HTTPS).
- Consultar domínios `.br` no **registro.br** só com a biblioteca padrão.
- Ler registros DNS (A, MX, NS, TXT) sem instalar nada.
- Calcular a idade do domínio do caso e cruzar com as denúncias.

## 🧠 Conceito em 2 minutos

Quando alguém compra um domínio, os dados do registro ficam guardados no **registro** daquela
terminação. Para `.br`, é o **registro.br** (NIC.br). Existem duas formas de perguntar:

| | WHOIS | RDAP |
|---|---|---|
| Transporte | TCP, porta 43 | HTTPS |
| Formato | texto livre, muda de servidor para servidor | JSON padronizado (RFC 9083) |
| Descoberta do servidor | perguntar a `whois.iana.org` | arquivo *bootstrap* da IANA |
| Melhor para | registros antigos, campos específicos do `.br` | automação e scripts |

**DNS** é outra coisa: é a "lista telefônica" que diz para onde o domínio aponta hoje (IP, servidor
de e-mail, servidores de nome). O **registro TXT com SPF** revela quais serviços podem enviar e-mail
em nome do domínio. É ótimo para descobrir o provedor de e-mail de uma empresa.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/dns_info.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-15-dominios/original/dns_info.py) | Pergunta o registro A (IP) de `osintme.com` com `dnspython` |
| [`original/whois_info.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-15-dominios/original/whois_info.py) | Consulta o WHOIS de `sector035.nl` com `python-whois` e mostra a data de criação |

```bash
pip install dnspython python-whois
cd dia-15-dominios
python original/dns_info.py
python original/whois_info.py
```

> 💡 Troque os domínios do autor por outros e compare. Experimente `python-whois` com um `.br`
> e veja quais campos ele consegue (ou não) separar.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q dnspython python-whois

### `original/dns_info.py` — Descobrir o IP (registro A) de um domínio com dnspython

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Descobrir o IP (registro A) de um domínio com dnspython
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_15/dns_info.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install dnspython
# ──────────────────────────────────────────────────────────────────────────

# 💬 "dns" é o pacote dnspython (não vem com o Python; precisa instalar).
import dns
# 💬 O módulo resolver faz as perguntas ao servidor DNS por nós.
import dns.resolver

# 💬 Pergunta: "qual é o registro A (endereço IPv4) de osintme.com?"
result = dns.resolver.resolve('osintme.com', 'A')
# 💬 Pode haver mais de um IP; percorremos todos.
for ipval in result:
    # 💬 to_text() transforma o objeto de resposta em texto legível.
    print('IP', ipval.to_text())

### `original/whois_info.py` — Consulta WHOIS de um domínio com python-whois

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Consulta WHOIS de um domínio com python-whois
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_15/whois_info.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install python-whois
# ──────────────────────────────────────────────────────────────────────────

# 💬 O pacote python-whois consulta o WHOIS e já entrega os campos separados.
import whois 

# 💬 Consulta o domínio sector035.nl (site de OSINT holandês, o alvo do autor).
whois_info = whois.whois('sector035.nl')

# 💬 Mostra tudo o que veio na resposta.
print (whois_info)

print("Creation date")

# 💬 A data de criação é um dos campos mais valiosos: domínio novo + promoção
# 💬 boa demais é um sinal clássico de golpe.
print(whois_info["creation_date"])

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão**. Cada script tem amostra offline em `dados/`.

| Arquivo | O que faz |
|---|---|
| [`br/01_rdap_br.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-15-dominios/br/01_rdap_br.py) | Descobre o servidor RDAP pelo bootstrap da IANA e consulta o domínio (padrão: `registro.br`) |
| [`br/02_whois_porta43.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-15-dominios/br/02_whois_porta43.py) | WHOIS "na unha" via `socket`, com descoberta do servidor na IANA |
| [`br/03_dns_doh.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-15-dominios/br/03_dns_doh.py) | IPs via `socket` e MX/NS/TXT via DNS sobre HTTPS, com leitura do SPF |
| [`br/04_caso_dominio_fantasma.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-15-dominios/br/04_caso_dominio_fantasma.py) | **O caso:** RDAP x WHOIS do domínio fictício e idade do site na 1ª denúncia |

```bash
cd dia-15-dominios
python br/01_rdap_br.py
OSINT_OFFLINE=1 python br/02_whois_porta43.py     # força o modo offline
```

**O que você deve ver no caso:** o domínio fictício foi criado em 02/10/2025 e fez a primeira vítima
**10 dias depois**. As duas fontes (RDAP e WHOIS) concordam na data.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_rdap_br.py` — RDAP: o "WHOIS moderno", em JSON, consultado só com a biblioteca padrão.

In [ ]:
%cd -q {PASTA_NB}
"""
RDAP: o "WHOIS moderno", em JSON, consultado só com a biblioteca padrão.

Carta 36 (rank A) — Jutsu do Registro Estruturado

O que você aprende:
  • o que é RDAP e por que ele está substituindo o WHOIS;
  • como descobrir QUAL servidor RDAP atende um domínio (bootstrap da IANA);
  • como ler datas de registro, alteração e expiração, servidores DNS e status.

Como rodar (dentro da pasta dia-15-dominios):
    python br/01_rdap_br.py
    OSINT_OFFLINE=1 python br/01_rdap_br.py   # sem internet, usa dados/
"""

import json                 # converter o texto JSON em dicionário Python
import os                   # ler a variável de ambiente OSINT_OFFLINE
import urllib.request       # fazer requisições HTTP sem instalar nada
from datetime import datetime, timezone  # calcular a idade do domínio
from pathlib import Path    # montar caminhos de arquivo

# ── Parâmetros (no Colab viram um formulário) ─────────────────────────────
DOMINIO = "registro.br"  #@param {type:"string"}

# ── Onde estão as amostras offline ────────────────────────────────────────
# Funciona rodando como script (python br/01_rdap_br.py) e como célula do Colab.
PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def buscar_json(url, amostra):
    """Baixa um JSON da internet; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            # Alguns servidores recusam pedidos sem User-Agent; nos identificamos.
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=15) as resposta:
                return json.load(resposta)
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    return json.loads((PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8"))


# ── Passo 1: descobrir o servidor RDAP certo ──────────────────────────────
# A IANA publica um arquivo "bootstrap" dizendo qual servidor responde por
# cada terminação (TLD). Ex.: ".br" → https://rdap.registro.br/
bootstrap = buscar_json("https://data.iana.org/rdap/dns.json", "iana_rdap_dns.json")

tld = DOMINIO.rsplit(".", 1)[-1].lower()   # "registro.br" → "br"
servidor = None
for terminacoes, urls in bootstrap["services"]:   # cada item: [[tlds], [urls]]
    if tld in terminacoes:
        servidor = urls[0]                         # pegamos o primeiro servidor
        break

if servidor is None:
    raise SystemExit(f"Nenhum servidor RDAP conhecido para .{tld}")

print(f"🔎 Servidor RDAP para .{tld}: {servidor}")

# ── Passo 2: consultar o domínio ──────────────────────────────────────────
# O caminho padrão do RDAP é /domain/<nome>. Garantimos uma barra entre eles.
url = servidor.rstrip("/") + "/domain/" + DOMINIO
dados = buscar_json(url, "rdap_registro_br.json")

print(f"\n🌐 Domínio: {dados.get('ldhName', DOMINIO)}")
print(f"📌 Status:  {', '.join(dados.get('status', []))}")

# ── Passo 3: eventos (datas) ──────────────────────────────────────────────
# "events" é uma lista de {eventAction, eventDate}. Guardamos num dicionário.
datas = {e["eventAction"]: e["eventDate"] for e in dados.get("events", [])}
for acao, quando in datas.items():
    print(f"🗓️  {acao:<14} → {quando}")

# ── Passo 4: idade do domínio (sinal de alerta clássico) ──────────────────
if "registration" in datas:
    # fromisoformat entende "2025-10-02T14:31:08Z" se trocarmos o Z por +00:00
    criado = datetime.fromisoformat(datas["registration"].replace("Z", "+00:00"))
    idade = (datetime.now(timezone.utc) - criado).days
    print(f"\n⏳ Idade do domínio: {idade} dias")
    if idade < 90:
        print("🚩 Domínio com menos de 90 dias: combine com outros sinais antes de concluir algo.")

# ── Passo 5: servidores DNS ───────────────────────────────────────────────
for ns in dados.get("nameservers", []):
    print(f"🧭 Nameserver: {ns.get('ldhName')}")

### `br/02_whois_porta43.py` — WHOIS "na unha": conversando direto com o servidor na porta 43 (só stdlib).

In [ ]:
%cd -q {PASTA_NB}
"""
WHOIS "na unha": conversando direto com o servidor na porta 43 (só stdlib).

Carta 36 (rank A) — Jutsu do Registro Estruturado (parte 2)

O que você aprende:
  • que o WHOIS é só uma conversa de texto numa porta TCP (a 43);
  • como perguntar à IANA qual servidor WHOIS cuida de cada terminação;
  • como transformar as linhas "campo: valor" em dicionário.

Por que manter WHOIS se existe RDAP? Porque muitos registros ainda respondem melhor
(ou só) por WHOIS, e porque o formato do registro.br traz campos úteis como
"owner", "owner-c" e "created". Analista bom sabe usar os dois.

Como rodar (dentro da pasta dia-15-dominios):
    python br/02_whois_porta43.py
    OSINT_OFFLINE=1 python br/02_whois_porta43.py
"""

import os                # ler OSINT_OFFLINE
import socket            # abrir uma conexão TCP crua (é tudo o que o WHOIS precisa)
from pathlib import Path

DOMINIO = "registro.br"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def whois(servidor, pergunta):
    """Abre a porta 43, envia a pergunta e devolve tudo o que o servidor responder."""
    with socket.create_connection((servidor, 43), timeout=15) as conexao:
        conexao.sendall((pergunta + "\r\n").encode())   # o protocolo pede \r\n no fim
        pedacos = []
        while True:
            pedaco = conexao.recv(4096)   # lemos de 4 em 4 KB...
            if not pedaco:                # ...até o servidor fechar a conexão
                break
            pedacos.append(pedaco)
    # Servidores antigos às vezes mandam latin-1; "replace" evita que o script quebre.
    return b"".join(pedacos).decode("utf-8", errors="replace")


def descobrir_servidor(dominio):
    """Pergunta à IANA (whois.iana.org) qual servidor WHOIS responde pela terminação."""
    tld = dominio.rsplit(".", 1)[-1]
    resposta = whois("whois.iana.org", tld)
    for linha in resposta.splitlines():
        if linha.lower().startswith("whois:"):          # ex.: "whois:  whois.registro.br"
            return linha.split(":", 1)[1].strip()
    return None


def em_dicionario(texto):
    """Converte linhas 'campo: valor' em {campo: [valores]} (campos podem repetir)."""
    campos = {}
    for linha in texto.splitlines():
        if ":" in linha and not linha.startswith("%"):  # "%" = comentário no registro.br
            chave, valor = linha.split(":", 1)
            campos.setdefault(chave.strip(), []).append(valor.strip())
    return campos


# ── Execução ──────────────────────────────────────────────────────────────
texto = None
if not OFFLINE:
    try:
        servidor = descobrir_servidor(DOMINIO)
        print(f"🔎 Servidor WHOIS indicado pela IANA: {servidor}")
        texto = whois(servidor, DOMINIO)
    except Exception as erro:
        print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
else:
    print("⚠️ MODO OFFLINE: usando a amostra local.")

if texto is None:
    texto = (PASTA_DIA / "dados" / "whois_registro_br.txt").read_text(encoding="utf-8")

campos = em_dicionario(texto)

# Mostramos só o que interessa numa investigação (o resto está em `texto`).
for chave in ["domain", "owner", "owner-c", "nserver", "created", "changed", "expires", "status"]:
    for valor in campos.get(chave, []):
        print(f"{chave:>8}: {valor}")

### `br/03_dns_doh.py` — DNS sem dnspython: socket (A/AAAA) + DNS sobre HTTPS (MX, NS, TXT).

In [ ]:
%cd -q {PASTA_NB}
"""
DNS sem dnspython: socket (A/AAAA) + DNS sobre HTTPS (MX, NS, TXT).

Carta 36 (rank A) — Jutsu do Registro Estruturado (parte 3)

O que você aprende:
  • a resolver IPs com socket.getaddrinfo (vem com o Python);
  • a pedir outros tipos de registro (MX, NS, TXT) a um resolvedor público via HTTPS;
  • a ler o SPF (registro TXT) para descobrir quem pode enviar e-mail pelo domínio.

Como rodar (dentro da pasta dia-15-dominios):
    python br/03_dns_doh.py
    OSINT_OFFLINE=1 python br/03_dns_doh.py
"""

import json
import os
import socket
import urllib.parse      # montar a URL com parâmetros (?name=...&type=...)
import urllib.request
from pathlib import Path

DOMINIO = "gov.br"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"

# Números oficiais de cada tipo de registro DNS (o resolvedor responde com eles).
TIPOS = {1: "A", 2: "NS", 5: "CNAME", 15: "MX", 16: "TXT", 28: "AAAA"}


def doh(nome, tipo):
    """Pergunta ao resolvedor público do Google via HTTPS. Devolve a lista 'Answer'."""
    url = "https://dns.google/resolve?" + urllib.parse.urlencode({"name": nome, "type": tipo})
    pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
    with urllib.request.urlopen(pedido, timeout=15) as resposta:
        return json.load(resposta).get("Answer", [])


# ── 1. IPs com socket (sem internet, isto falha, então tratamos o erro) ───
print(f"🌐 Domínio: {DOMINIO}\n")
if not OFFLINE:
    try:
        ips = {info[4][0] for info in socket.getaddrinfo(DOMINIO, None)}  # conjunto: sem repetidos
        for ip in sorted(ips):
            print(f"📍 IP: {ip}")
    except socket.gaierror as erro:
        print(f"⚠️ MODO OFFLINE: não foi possível resolver ({erro}).")

# ── 2. MX, NS e TXT via DNS sobre HTTPS ───────────────────────────────────
respostas = None
if not OFFLINE:
    try:
        respostas = {tipo: doh(DOMINIO, tipo) for tipo in ["MX", "NS", "TXT"]}
    except Exception as erro:
        print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
else:
    print("⚠️ MODO OFFLINE: usando a amostra local.")

if respostas is None:
    amostra = json.loads((PASTA_DIA / "dados" / "doh_gov_br.json").read_text(encoding="utf-8"))
    respostas = amostra["respostas"]

for tipo, lista in respostas.items():
    print(f"\n── {tipo} ──")
    for item in lista:
        print(f"  {TIPOS.get(item['type'], item['type']):<5} {item['data']}")

# ── 3. Bônus: o SPF entrega os provedores de e-mail usados ────────────────
for item in respostas.get("TXT", []):
    if "v=spf1" in item["data"]:
        includes = [p.split(":", 1)[1] for p in item["data"].strip('"').split() if p.startswith("include:")]
        print(f"\n✉️  SPF autoriza envio por: {', '.join(includes) or '(nenhum include)'}")

### `br/04_caso_dominio_fantasma.py` — Caso Vitrine Fantasma: o domínio da loja falsa nasceu quando?

In [ ]:
%cd -q {PASTA_NB}
"""
Caso Vitrine Fantasma: o domínio da loja falsa nasceu quando?

Carta 36 (rank A) — Jutsu do Registro Estruturado (aplicação no caso)

O que você aprende:
  • a cruzar a data de criação do domínio (RDAP/WHOIS) com as datas das denúncias;
  • a medir "quantos dias de vida o site tinha quando fez a primeira vítima";
  • a comparar RDAP x WHOIS: as duas fontes concordam?

Este script é 100% offline: usa os dados fictícios de caso/dados/.

Como rodar (dentro da pasta dia-15-dominios):
    python br/04_caso_dominio_fantasma.py
"""

import csv
import json
from datetime import date, datetime
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"

# ── 1. O que diz o RDAP ───────────────────────────────────────────────────
rdap = json.loads((PASTA_CASO / "rdap_megaoferta.json").read_text(encoding="utf-8"))
eventos = {e["eventAction"]: e["eventDate"] for e in rdap["events"]}
criado_rdap = datetime.fromisoformat(eventos["registration"].replace("Z", "+00:00")).date()

# ── 2. O que diz o WHOIS (formato registro.br: created: AAAAMMDD #id) ─────
criado_whois = None
for linha in (PASTA_CASO / "whois_megaoferta.txt").read_text(encoding="utf-8").splitlines():
    if linha.startswith("created:"):
        aaaammdd = linha.split(":", 1)[1].split()[0]           # "20251002"
        criado_whois = datetime.strptime(aaaammdd, "%Y%m%d").date()

print(f"🌐 Domínio investigado: {rdap['ldhName']}")
print(f"🗓️  Criação segundo o RDAP : {criado_rdap:%d/%m/%Y}")
print(f"🗓️  Criação segundo o WHOIS: {criado_whois:%d/%m/%Y}")
print("✅ As fontes concordam." if criado_rdap == criado_whois else "🚩 As fontes DISCORDAM: anote no relatório!")

# ── 3. A primeira e a última denúncia ─────────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    datas = [date.fromisoformat(linha["data"]) for linha in csv.DictReader(arquivo, delimiter=";")]

primeira, ultima = min(datas), max(datas)
print(f"\n📨 Primeira denúncia: {primeira:%d/%m/%Y}  ·  Última: {ultima:%d/%m/%Y}")
print(f"⏳ O site tinha {(primeira - criado_rdap).days} dias de vida na primeira vítima.")
print(f"⏳ E {(ultima - criado_rdap).days} dias na última.")

# ── 4. Quem aparece como titular? ─────────────────────────────────────────
for entidade in rdap.get("entities", []):
    if "registrant" in entidade.get("roles", []):
        # vCard é uma lista de listas: procuramos o campo "fn" (nome completo)
        nome = next((c[3] for c in entidade["vcardArray"][1] if c[0] == "fn"), "?")
        print(f"\n👤 Titular declarado no registro: {nome} (handle {entidade.get('handle')})")
        print("   Lembre-se: dado de registro pode ser falso ou de laranja. É pista, não prova.")

## ⚖️ Ética, LGPD e OPSEC

- Dados de registro de pessoa física no `.br` são **dados pessoais** (LGPD). Use só o necessário para
  a finalidade da investigação e registre de onde veio cada informação.
- O titular declarado pode ser **falso ou um laranja**. Registro é pista, nunca prova sozinho.
- O registro.br limita consultas. Não faça varreduras em massa: respeite os limites e o termo de uso.
- Consultas WHOIS/RDAP **não alertam** o dono do domínio. Já acessar o site do suspeito pode deixar
  rastro (seu IP nos logs dele). Veja o Dia 12.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Qual a principal vantagem do RDAP sobre o WHOIS tradicional?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) É mais antigo e mais estável", "B) Devolve dados estruturados em JSON via HTTPS", "C) Mostra a senha do painel do domínio", "D) Só funciona para domínios .br"]
correta = "B) Devolve dados estruturados em JSON via HTTPS"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "RDAP padroniza a resposta em JSON (RFC 9083) e usa HTTPS, o que facilita a automação.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Como o script descobre qual servidor RDAP atende o domínio?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Chuta rdap.<tld>.com", "B) Pergunta ao Google", "C) Lê o arquivo bootstrap da IANA (dns.json)", "D) Usa o registro MX do domínio"]
correta = "C) Lê o arquivo bootstrap da IANA (dns.json)"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "A IANA publica em data.iana.org/rdap/dns.json o mapa TLD → servidor RDAP.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. O registro TXT com 'v=spf1 include:spf.protection.outlook.com' indica que…
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) o domínio usa Microsoft 365 para enviar e-mail", "B) o site está hospedado na Microsoft", "C) o domínio foi registrado pela Microsoft", "D) o domínio está bloqueado"]
correta = "A) o domínio usa Microsoft 365 para enviar e-mail"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O SPF lista quem pode enviar e-mail em nome do domínio; esse include é do Microsoft 365/Outlook.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 15: Radar de domínios recém-nascidos

**Carta 57 · Rank S**

## A missão

A Iara quer um **radar**: sempre que surgir uma nova denúncia de loja falsa, o NFA cola o domínio numa
lista e o script diz, em segundos, quais são "recém-nascidos".

Escreva um script que:

1. Receba uma lista de domínios separados por vírgula (parâmetro `DOMINIOS`).
2. Descubra o servidor RDAP de cada um pelo **bootstrap da IANA**.
3. Consulte o RDAP e extraia: data de criação, idade em dias e nameservers.
4. Marque com 🚩 quem tiver menos de `LIMITE_DIAS` (padrão: 90).
5. Grave `saidas/radar_dominios.csv` com separador `;` e codificação `utf-8-sig`
   (assim o Excel em português abre certinho).
6. Funcione **sem internet** usando as amostras (`dados/` e `caso/dados/`).

## Critérios de rank S

- [ ] Só biblioteca padrão
- [ ] Não quebra se um domínio não existir ou o servidor não responder
- [ ] O domínio do caso aparece como 🚩
- [ ] CSV abre no Excel sem caracteres estranhos

## Dica

Monte um dicionário `tld → servidor` uma única vez, antes do laço. Assim você baixa o bootstrap só uma vez.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 15 — Radar de domínios recém-nascidos.

Lê uma lista de domínios, consulta o RDAP de cada um (descobrindo o servidor pelo
bootstrap da IANA) e grava radar_dominios.csv com idade e um alerta.
Sem internet (ou com OSINT_OFFLINE=1), usa as amostras locais.
"""

import csv
import json
import os
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

DOMINIOS = "registro.br, nic.br, megaoferta-brasil.example"  #@param {type:"string"}
LIMITE_DIAS = 90  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"

# Amostras locais por domínio (usadas quando não há internet)
AMOSTRAS = {
    "registro.br": PASTA_DIA / "dados" / "rdap_registro_br.json",
    "megaoferta-brasil.example": PASTA_CASO / "rdap_megaoferta.json",
}


def baixar(url):
    pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
    with urllib.request.urlopen(pedido, timeout=15) as resposta:
        return json.load(resposta)


def servidores_rdap():
    """Dicionário tld → url do servidor RDAP."""
    try:
        if OFFLINE:
            raise RuntimeError("offline")
        bootstrap = baixar("https://data.iana.org/rdap/dns.json")
    except Exception:
        bootstrap = json.loads((PASTA_DIA / "dados" / "iana_rdap_dns.json").read_text(encoding="utf-8"))
    return {tld: urls[0] for tlds, urls in bootstrap["services"] for tld in tlds}


def consultar(dominio, mapa):
    tld = dominio.rsplit(".", 1)[-1]
    if not OFFLINE and tld in mapa:
        try:
            return baixar(mapa[tld].rstrip("/") + "/domain/" + dominio), "ao vivo"
        except Exception:
            pass
    if dominio in AMOSTRAS:
        return json.loads(AMOSTRAS[dominio].read_text(encoding="utf-8")), "amostra"
    return None, "sem dados"


mapa = servidores_rdap()
linhas = []
for dominio in [d.strip() for d in DOMINIOS.split(",") if d.strip()]:
    dados, origem = consultar(dominio, mapa)
    linha = {"dominio": dominio, "origem": origem, "criado_em": "", "idade_dias": "",
             "nameservers": "", "alerta": ""}
    if dados:
        eventos = {e["eventAction"]: e["eventDate"] for e in dados.get("events", [])}
        if "registration" in eventos:
            criado = datetime.fromisoformat(eventos["registration"].replace("Z", "+00:00"))
            # Para a amostra fictícia, medimos a idade na data da 1ª denúncia (12/10/2025)
            referencia = datetime(2025, 10, 12, tzinfo=timezone.utc) if dominio.endswith(".example") \
                else datetime.now(timezone.utc)
            idade = (referencia - criado).days
            linha.update(criado_em=criado.date().isoformat(), idade_dias=idade,
                         alerta="🚩 recém-nascido" if idade < LIMITE_DIAS else "ok")
        linha["nameservers"] = " ".join(ns.get("ldhName", "") for ns in dados.get("nameservers", []))
    linhas.append(linha)
    print(f"{dominio:<28} {origem:<9} {linha['idade_dias']!s:>6} dias  {linha['alerta']}")

SAIDAS = PASTA_DIA / "saidas"
SAIDAS.mkdir(exist_ok=True)
with open(SAIDAS / "radar_dominios.csv", "w", newline="", encoding="utf-8-sig") as f:
    escritor = csv.DictWriter(f, fieldnames=list(linhas[0]), delimiter=";")
    escritor.writeheader()
    escritor.writerows(linhas)
print("\n💾 saidas/radar_dominios.csv gravado (abre direto no Excel em pt-BR).")

## 🔎 Inovação OSINT do dia

**RDAP com bootstrap automático.** Em vez de decorar qual servidor atende cada terminação, o script
lê `https://data.iana.org/rdap/dns.json` e descobre sozinho. A ICANN encerrou a obrigação de WHOIS
para os gTLDs (como `.com`) em janeiro de 2025, então RDAP virou o caminho oficial para eles. No `.br`,
os dois continuam funcionando. Biblioteca Python simples: `urllib.request` + `json` (nativos).

## 📚 Para ir além

- RDAP do registro.br: <https://rdap.registro.br/>
- Bootstrap RDAP da IANA: <https://data.iana.org/rdap/>
- RFC 9083 (formato de resposta RDAP): <https://www.rfc-editor.org/rfc/rfc9083>
- DNS sobre HTTPS do Google (formato JSON): <https://developers.google.com/speed/public-dns/docs/doh/json>

## ✅ Checklist

- [ ] Rodei os dois scripts originais
- [ ] Consultei um domínio `.br` por RDAP **e** por WHOIS e comparei as datas
- [ ] Descobri o provedor de e-mail de um domínio pelo SPF
- [ ] Calculei a idade do domínio do caso na primeira denúncia
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 15](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)